1. Importing Necessary Libraries

In [ ]:
import os, json, random, math
import numpy as np
import pandas as pd
from collections import deque

import tensorflow as tf
from tensorflow.keras import layers as L, models as KM

from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve, auc
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(tf.__version__)

2.20.0


2. Configuration: Paths, Hyperparameters, and Schedules for Lifelong SSL Adaptation

In [ ]:
# This block centralizes all runtime settings (paths, batch sizes, schedules, and optimizer/loss).
# Adjust only the values below; the rest of the pipeline should read from these constants.

# Data / Model paths 
CSV_PATH = 'train_semisupervised_detector_annotations.csv'      # CSV with columns: image_id, adv_label, isNight, split 
MODEL_PATH = 'models/model_v_semi.keras'                        # Pretrained student model (.keras)
IMG_ROOT = '.'                                                  # Prefix to resolve relative image paths (if needed)

# Core runtime params 
IMG_SIZE   = 50                                                 # Image resize (HxW)
BATCH_L    = 64                                                 # Labeled batch size
BATCH_U    = 256                                                # Unlabeled batch size
EPOCHS_ADAPT_PER_EP = 4                                         # Local epochs per episode during adaptation
LR         = 2e-4                                               # Learning rate for adaptation (keep modest to avoid drift)

# Semi-supervised schedules 
TAU_START     = 0.90                                            # Pseudo-label confidence threshold (warmup)
TAU_END       = 0.95                                            # Stricter threshold after warmup
LAMBDA_U_MAX  = 1.5                                             # Max weight for unsupervised loss
WARMUP_EPOCHS = 1                                               # Epochs with λ_u ramping from 0 to LAMBDA_U_MAX
TEMP          = 0.7                                             # (If used) temperature for sharpening / soft pseudo-labeling

# EMA teacher parameters 
EMA_ALPHA_WARM = 0.98                                         # Faster EMA at the start for quick stabilization
EMA_ALPHA_LATE = 0.999                                        # Slower EMA later for smoother updates

# Replay & Learning without Forgetting (LwF) 
REPLAY_CAPACITY = 4000                                        # Max items kept in the replay buffer
REPLAY_FRACTION = 0.2                                         # Fraction of a batch sampled from replay
LAMBDA_REPLAY   = 0.5                                         # Weight of replay loss term
USE_LWF         = True                                        # Enable/distable LwF regularization
LAMBDA_LWF      = 0.2                                         # Weight of LwF distillation loss

# Episodic setup & splits (lifelong learning) 
NUM_EPISODES    = 4                                           # Number of adaptation episodes+
VAL_SPLIT_NAME  = 'val'                                       # Name of validation split in CSV
TEST_SPLIT_NAME = 'test'                                      # Name of test split in CSV

# Optimizer & loss (assumes model outputs probabilities via sigmoid) 
optimizer = tf.keras.optimizers.Adam(LR)                      # Global optimizer (keep outside train_step to preserve state)
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)   # Set to from_logits=True if your model outputs raw logits

3. Data Ingestion & Split Extraction

In [ ]:
# Loads the unified annotation CSV, normalizes paths, and prepares
# labeled/unlabeled pools along with validation and test splits.

# Expected columns: image_id, adv_label (0/1), isNight (0/1), split (train/val/test)
df = pd.read_csv(CSV_PATH)

# Sanity check: fail fast if required columns are missing
assert {'image_id','adv_label','isNight','split'}.issubset(set(df.columns)), df.columns

def join_root(p):
    """
    Prefix relative image paths with IMG_ROOT (when provided),
    leaving absolute or already-rooted paths untouched.
    """
    if isinstance(p, str) and IMG_ROOT not in ['', '.'] and not p.startswith(IMG_ROOT):
        return os.path.join(IMG_ROOT, p)
    return p

# Normalize image paths
df['image_id'] = df['image_id'].apply(join_root)

# Labeled pool: rows with available labels within train/val splits
labeled_df = df[df['split'].isin(['train', 'val']) & df['adv_label'].notna()].copy()

# Validation & test frames (split names are configurable via VAL_SPLIT_NAME / TEST_SPLIT_NAME)
val_df     = df[df['split'] == VAL_SPLIT_NAME].copy()
test_df    = df[df['split'] == TEST_SPLIT_NAME].copy()

# Unlabeled candidate pool: all train/val rows (labels may be missing/ignored for SSL)
unlabeled_pool = df[df['split'].isin(['train', 'val'])].copy()

print('Labeled:', len(labeled_df), 'Unlabeled pool:', len(unlabeled_pool), 'Val:', len(val_df), 'Test:', len(test_df))

Labeled: 120681 Unlabeled pool: 120681 Val: 26751 Test: 34797


4. tf.data Pipelines: Decoding, Augmentations, and Dataset Builders

In [ ]:
# Provides efficient input pipelines for labeled and unlabeled data with weak/strong augmentations.

AUTOTUNE = tf.data.AUTOTUNE

def decode_img(path):
    """
    Read and decode an image from disk, resize to IMG_SIZE, and normalize to [0, 1].

    Parameters
    ----------
    path : tf.Tensor (scalar string)
        Filepath to an image (JPEG expected).

    Returns
    -------
    tf.Tensor
        Float32 tensor of shape (IMG_SIZE, IMG_SIZE, 3) in [0, 1].
    """
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img

def aug_weak(img):
    """
    Lightweight augmentation preserving semantics: horizontal flip.
    """
    img = tf.image.random_flip_left_right(img)
    return img

def aug_strong(img):
    """
    Stronger augmentation for FixMatch-style SSL: flip + brightness/contrast jitter.
    """
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def build_labeled_ds(frame, shuffle=True, repeat=False, batch=64):
    """
    Build a labeled dataset yielding ((image, isNight), label) batches.

    Notes
    -----
    - Expects DataFrame columns: image_id, adv_label, isNight.
    - Uses weak augmentation to avoid over-regularizing the supervised signal.
    """
    paths = tf.constant(frame['image_id'].values)
    y     = tf.constant(frame['adv_label'].values, dtype=tf.float32)
    meta  = tf.constant(frame['isNight'].values, dtype=tf.float32)
    def _map(i, m, t):
        img = decode_img(i)
        img = aug_weak(img)
        m   = tf.reshape(m, [1])
        t   = tf.reshape(t, [1])
        return ((img, m), t)
    ds = tf.data.Dataset.from_tensor_slices((paths, meta, y))
    if shuffle: ds = ds.shuffle(len(frame), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    if repeat: ds = ds.repeat()
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

def build_unlabeled_ds(frame, batch=256):
    """
    Build an unlabeled dataset yielding ((img_weak, img_strong, isNight), dummy_label) batches.

    Notes
    -----
    - Weak/strong views are used for consistency training (FixMatch).
    - The dummy label is a placeholder to keep a uniform signature.
    """
    paths = tf.constant(frame['image_id'].values)
    meta  = tf.constant(frame['isNight'].values, dtype=tf.float32)
    def _map(i, m):
        img = decode_img(i)
        iw  = aug_weak(img)
        is_ = aug_strong(img)
        m   = tf.reshape(m, [1])
        return ((iw, is_, m), tf.zeros([1], tf.float32))
    ds = tf.data.Dataset.from_tensor_slices((paths, meta))
    ds = ds.shuffle(len(frame), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

5. Model: Binary CNN with Meta-Feature + EMA Utilities

In [ ]:
# Defines a lightweight CNN classifier that ingests an image plus a 1-D meta feature (isNight),
# along with helpers to (a) clone a Keras model including weights and (b) update an EMA teacher.

def build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """
    Build a compact binary CNN that fuses an image embedding with a scalar meta feature.

    Parameters
    ----------
    input_shape : tuple
        Shape of the image input (H, W, C).

    Returns
    -------
    tf.keras.Model
        A model with two inputs: 
          - image: (H, W, C), float32 in [0, 1]
          - isNight: (1,), scalar meta feature
        and one output:
          - probability (sigmoid) for the "adversarial" class.
    """
    img_in  = L.Input(shape=input_shape, name='image')
    meta_in = L.Input(shape=(1,), name='isNight')

    # Image tower 
    x = img_in
    x = L.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = L.MaxPool2D()(x)
    x = L.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = L.MaxPool2D()(x)
    x = L.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = L.GlobalAveragePooling2D()(x)

    # Feature fusion 
    z = L.Concatenate()([x, meta_in])           # append scalar meta-feature
    z = L.Dense(128, activation='relu')(z)
    z = L.Dropout(0.3)(z)

    # Binary head 
    out = L.Dense(1, activation='sigmoid')(z)   # prob in (0,1)

    return KM.Model(inputs=[img_in, meta_in], outputs=out, name='cnn_bin')

def clone_model_with_weights(model):
    """
    Clone a Keras model and copy its weights (useful for snapshotting student→teacher).

    Notes
    -----
    - Ensures the cloned model is built before setting weights.
    """
    m = tf.keras.models.clone_model(model)
    m.build(model.inputs[0].shape)          # build with same input signature
    m.set_weights(model.get_weights())      # copy weights 1:1
    return m

def update_ema(teacher, student, alpha):
    """
    Exponential Moving Average (EMA) update for the teacher's weights.

    Parameters
    ----------
    teacher : tf.keras.Model
        The EMA model to be updated in-place.
    student : tf.keras.Model
        The current student model providing fresh weights.
    alpha : float
        EMA factor; higher values (e.g., 0.999) yield smoother, more stable teachers.

    Formula
    -------
    w_teacher ← α * w_teacher + (1 - α) * w_student
    """
    st_w = student.get_weights()
    te_w = teacher.get_weights()
    teacher.set_weights([alpha*tw + (1.0-alpha)*sw for tw,sw in zip(te_w, st_w)])

6. Model Loading: Restore Pretrained Student and Initialize EMA Teacher

In [ ]:
# Tries to load a fully-saved Keras model first; if that fails, builds the CNN and
# loads weights only. Then clones the student to create the teacher (EMA).

try:
    # Preferred path: full model file (.keras / SavedModel dir)
    student = tf.keras.models.load_model(MODEL_PATH)
except Exception as e:
    print('Falling back to build and load weights:', e)
    # Fallback: instantiate the architecture and load only the weights
    student = build_cnn_binary()
    student.load_weights(MODEL_PATH)

# Initialize teacher as a weight-identical clone of student (for EMA updates)
teacher = tf.keras.models.clone_model(student)
teacher.build(student.inputs[0].shape)      # ensure layers are built before setting weights
teacher.set_weights(student.get_weights())

print('Student and teacher ready.')

Student and teacher ready.


7. Evaluation Utilities: Batched Inference, Metrics, and Frame-Level Reporting

In [ ]:
# Provides fast batched prediction, ROC/PR metrics, FPR@TPR=95%, and Expected Calibration Error (ECE).

def predict_proba(model, frame, batch=256):
    """
    Run batched inference and return predicted probabilities in [0, 1].

    Parameters
    ----------
    model : tf.keras.Model
        Binary classifier that consumes (image, isNight) and outputs sigmoid probs.
    frame : pd.DataFrame
        Must contain columns: 'image_id', 'isNight'.
    batch : int
        Batch size used for tf.data pipeline during inference.

    Returns
    -------
    np.ndarray
        1D array of probabilities aligned with frame rows.
    """
    paths = tf.constant(frame['image_id'].values)
    meta  = tf.constant(frame['isNight'].values, dtype=tf.float32)

    def _map(i, m):
        img = decode_img(i)         # decode/resize/normalize
        m   = tf.reshape(m, [1])    # keep meta as (1,)
        return (img, m)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, meta))
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)

    preds = []
    for img, m in ds:
        p = model([img, m], training=False).numpy().ravel()
        preds.append(p)
    return np.concatenate(preds, axis=0)

def fpr_at_tpr95(y_true, y_score):
    """
    Compute FPR at exactly 95% TPR.

    Notes
    -----
    Uses nearest-neighbor to 0.95 TPR. For smoother estimates, consider
    a linear interpolation between ROC points (see optional version below).
    """
    fpr, tpr, thr = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(tpr - 0.95))
    return float(fpr[idx])

def expected_calibration_error(y_true, y_prob, n_bins=15):
    """
    Compute Expected Calibration Error (ECE) with equal-width bins.

    Parameters
    ----------
    y_true : array-like of shape (N,)
        Binary ground truth labels {0,1}.
    y_prob : array-like of shape (N,)
        Predicted probabilities in [0, 1].
    n_bins : int
        Number of bins for calibration histogram.

    Returns
    -------
    float
        ECE value (lower is better; 0 indicates perfect calibration).
    """
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if np.any(mask):
            conf = y_prob[mask].mean()
            acc  = ( (y_prob[mask] >= 0.5).astype(int) == y_true[mask] ).mean()
            ece += (mask.mean()) * abs(acc - conf)
    return float(ece)

def evaluate_frame(model, frame, tag='eval'):
    """
    Compute AUROC, PR-AUC, FPR@TPR=95%, and ECE on a given DataFrame subset.

    Parameters
    ----------
    model : tf.keras.Model
        Binary classifier with sigmoid output.
    frame : pd.DataFrame
        Must include: 'image_id', 'isNight', 'adv_label'.
    tag : str
        Label for logging/record-keeping (e.g., 'test_ep1_adapted').

    Returns
    -------
    dict
        { 'tag', 'AUROC', 'PR-AUC', 'FPR@TPR=95%', 'ECE' }
    """
    y_true = frame['adv_label'].astype(int).values
    y_prob = predict_proba(model, frame)

    # ROC/PR metrics
    auroc = roc_auc_score(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)

    # Operating-point and calibration
    fpr95 = fpr_at_tpr95(y_true, y_prob)
    ece = expected_calibration_error(y_true, y_prob)
    
    metrics = {'tag': tag, 'AUROC': auroc, 'PR-AUC': pr_auc, 'FPR@TPR=95%': fpr95, 'ECE': ece}
    print(metrics)
    return metrics

8. Episodic Split: Shuffle Unlabeled Pool and Partition into K Episodes

In [ ]:
# We randomly permute the unlabeled pool (seeded for reproducibility) and split it
# into NUM_EPISODES contiguous chunks of (roughly) equal size.

unlabeled_pool_shuffled = unlabeled_pool.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Partition into K episodes (np.array_split balances remainder across chunks)
episodes = np.array_split(unlabeled_pool_shuffled, NUM_EPISODES)

# Quick sanity check: sizes per episode
[len(ep) for ep in episodes]

c:\Users\evri\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[30171, 30170, 30170, 30170]

9. Replay Buffer: Storage, Sampling, and tf.data Builder

In [ ]:
# Maintains a bounded memory of past (image, meta, label) triples to mitigate forgetting.
# After each episode, we push confident pseudo-labeled samples; during training, we mix
# a fraction of replay batches into the current episode's updates.

class ReplayBuffer:
    def __init__(self, capacity=2000):
        """
        Parameters
        ----------
        capacity : int
            Maximum number of (image_id, isNight, adv_label) items to retain.
        """
        self.capacity = capacity
        self.items = deque(maxlen=capacity)
    def add_many(self, df_like):
        """
        Append many rows from a DataFrame-like object to the buffer.

        Expected columns: ['image_id', 'isNight', 'adv_label']
        """
        for _, r in df_like.iterrows():
            self.items.append((r['image_id'], r['isNight'], r['adv_label']))
    def sample(self, k=512):
        """
        Randomly sample up to k items from the buffer.

        Returns
        -------
        pd.DataFrame or None
            DataFrame with columns ['image_id','isNight','adv_label'] or None if empty.
        """
        k = min(k, len(self.items))
        if k == 0:
            return None
        idx = np.random.choice(len(self.items), k, replace=False)
        rows = [self.items[i] for i in idx]
        return pd.DataFrame(rows, columns=['image_id','isNight','adv_label'])

# Instantiate global replay buffer with configured capacity
replay = ReplayBuffer(capacity=REPLAY_CAPACITY)

def build_replay_ds(df_like, batch=128):
    """
    Construct a tf.data pipeline from replay samples.

    Parameters
    ----------
    df_like : pd.DataFrame or None
        Replay samples; if empty/None, returns None.
    batch : int
        Batch size for the replay dataset.

    Returns
    -------
    tf.data.Dataset or None
        Dataset yielding ((image, isNight), label) batches or None if no data.
    """
    if df_like is None or len(df_like)==0:
        return None
    
    paths = tf.constant(df_like['image_id'].values)
    meta  = tf.constant(df_like['isNight'].values, dtype=tf.float32)
    y     = tf.constant(df_like['adv_label'].values, dtype=tf.float32)

    def _map(i, m, t):
        img = decode_img(i)
        m   = tf.reshape(m, [1])
        t   = tf.reshape(t, [1])
        return ((img, m), t)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, meta, y))
    ds = ds.shuffle(len(df_like), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

10. Training Schedules and One-Step Update (FixMatch + EMA + Replay + LwF)

In [ ]:
# Provides small helpers for time-varying thresholds/weights and a single training step
# that combines supervised loss, unsupervised consistency, replay, and LwF regularization.

def schedule_tau(local_epoch):
    """
    Confidence threshold for pseudo-labels.
    Lower at the beginning (more pseudo-labels), stricter later (higher precision).
    """
    return TAU_START if local_epoch < 2 else TAU_END

def schedule_lambda_u(local_epoch):
    """
    Weight for the unsupervised loss (λ_u).
    Warm up from 0 to LAMBDA_U_MAX after WARMUP_EPOCHS to avoid instability early on.
    """
    return 0.0 if local_epoch < WARMUP_EPOCHS else LAMBDA_U_MAX

def schedule_ema(local_epoch):
    """
    EMA momentum (α) for the teacher.
    Faster averaging early, slower later for smoother targets.
    """
    return EMA_ALPHA_WARM if local_epoch < 2 else EMA_ALPHA_LATE

@tf.function
def train_step(student, teacher, batch_l, batch_u, tau, lambda_u, lambda_replay,
               use_lwf=False, teacher_prev=None, replay_batch=None):
    """
    Perform one optimization step:
      1) Supervised BCE on labeled batch.
      2) Unsupervised consistency (FixMatch) on unlabeled batch using teacher pseudo-labels.
      3) Optional replay BCE to mitigate forgetting.
      4) Optional LwF (distillation) against a frozen snapshot (teacher_prev).

    Parameters
    ----------
    student : tf.keras.Model
        The trainable model updated by gradients.
    teacher : tf.keras.Model
        EMA teacher providing stable pseudo-labels.
    batch_l : tuple
        ((img_l, meta_l), y_l) labeled batch.
          - img_l: [B_l, H, W, 3], meta_l: [B_l, 1], y_l: [B_l, 1]
    batch_u : tuple
        ((img_w, img_s, meta_u), _dummy) unlabeled batch with weak/strong views.
          - img_w/img_s: [B_u, H, W, 3], meta_u: [B_u, 1]
    tau : float
        Confidence threshold for pseudo-label filtering.
    lambda_u : float
        Weight for unsupervised loss term.
    lambda_replay : float
        Weight for replay loss term.
    use_lwf : bool
        If True, include LwF regularization.
    teacher_prev : tf.keras.Model or None
        Frozen snapshot for LwF (previous teacher/student).
    replay_batch : tuple or None
        ((img_r, meta_r), y_r) replay batch; may be None if buffer is empty.

    Returns
    -------
    dict
        Loss components for logging: {'loss_sup', 'loss_unsup', 'loss_replay', 'loss_lwf'}.
    """
    (img_l, meta_l), y_l = batch_l
    (img_w, img_s, meta_u), _ = batch_u

    with tf.GradientTape() as tape:
        # 1) Supervised BCE on labeled data
        p_l = student([img_l, meta_l], training=True)
        loss_sup = bce(y_l, p_l)

        # 2) Unsupervised consistency (FixMatch):
        #    - Teacher predicts on weakly augmented images
        #    - Keep only predictions above tau (confidence mask)
        #    - Student matches pseudo-labels on strongly augmented images
        p_w_teacher = teacher([img_w, meta_u], training=False)  # probs in [0,1]
        conf = tf.squeeze(p_w_teacher, axis=-1)                 # shape [B_u]
        mask = tf.cast(conf >= tau, tf.float32)                 # binary mask [B_u]
        y_pseudo = tf.cast(p_w_teacher >= 0.5, tf.float32)      # hard pseudo-labels [B_u,1]

        p_s = student([img_s, meta_u], training=True)
        loss_unsup_all = bce(y_pseudo, p_s)
        # Normalize by number of confident samples (avoid div-by-zero via epsilon)
        loss_unsup = tf.reduce_sum(loss_unsup_all * tf.reshape(mask, [-1,1])) / (tf.reduce_sum(mask) + 1e-6)

        # 3) Replay loss (optional): supervised BCE on memory samples
        loss_replay = 0.0
        if (replay_batch is not None) and (lambda_replay > 0.0):
            (img_r, meta_r), y_r = replay_batch
            p_r = student([img_r, meta_r], training=True)
            loss_replay = bce(y_r, p_r)

        # 4) Learning without Forgetting (optional): distill toward a frozen snapshot
        loss_lwf = 0.0
        if use_lwf and (teacher_prev is not None):
            # KL on probabilities (works but may be sensitive with 1D sigmoids).
            # For extra stability, consider L2 on logits instead.
            p_prev = tf.stop_gradient(teacher_prev([img_l, meta_l], training=False))
            loss_lwf = tf.reduce_mean(tf.keras.losses.KLDivergence(reduction='none')(p_prev, p_l))

        # Total loss (weighted sum of components)
        loss = loss_sup + lambda_u*loss_unsup + lambda_replay*loss_replay + (LAMBDA_LWF*loss_lwf if use_lwf else 0.0)

    # Backprop through student parameters
    grads = tape.gradient(loss, student.trainable_variables)
    optimizer.apply_gradients(zip(grads, student.trainable_variables))
    
    return {'loss_sup': loss_sup, 'loss_unsup': loss_unsup, 'loss_replay': loss_replay, 'loss_lwf': loss_lwf}

11. Confident Pseudo-Label Collection (Teacher-Guided)

In [ ]:
# Selects unlabeled samples whose predicted probability exceeds a confidence
# threshold (tau) and assigns hard pseudo-labels for subsequent replay/training.

def collect_confident_pseudolabels(model, frame, tau=0.95):
    """
    Collect confident pseudo-labeled samples from a DataFrame.

    Parameters
    ----------
    model : tf.keras.Model
        Model used to generate probabilities in [0, 1].
    frame : pd.DataFrame
        Must contain columns: 'image_id', 'isNight' (meta). Labels are not required.
    tau : float
        Confidence threshold; only samples with p >= tau are retained.

    Returns
    -------
    pd.DataFrame
        Subset with columns ['image_id', 'isNight', 'adv_label'] where adv_label
        is a hard pseudo-label (0/1). Index is reset.
    """
    # Predict probabilities for all rows in the frame
    p = predict_proba(model, frame)
    
    # Keep only highly confident predictions (teacher confidence filtering)
    mask = p >= tau

    # Carry over paths + meta; we will assign hard pseudo-labels next
    sub = frame.loc[mask, ['image_id','isNight']].copy()

    # Convert confident probabilities to hard labels at 0.5 threshold
    # (Note: classification threshold is 0.5; confidence gate is 'tau'.)
    sub['adv_label'] = (p[mask] >= 0.5).astype(np.float32)
    
    return sub.reset_index(drop=True)

12. Episodic Lifelong Adaptation Loop

In [ ]:
# For each episode:
#   1) Evaluate zero-shot performance on the held-out test set.
#   2) Adapt the student with supervised + SSL (FixMatch), optional Replay & LwF.
#   3) Update the EMA teacher after each step.
#   4) Evaluate again (post-adaptation), collect confident pseudo-labels into replay,
#      and checkpoint both student and teacher.

history = []

# Baseline validation before any adaptation
if len(val_df):
    history.append(evaluate_frame(student, val_df, tag='val_before_ep1'))

# Shuffle unlabeled pool and split into episodes
unlabeled_pool_shuffled = unlabeled_pool.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
episodes = np.array_split(unlabeled_pool_shuffled, NUM_EPISODES)

for e_idx, epi_df in enumerate(episodes, 1):
    print(f"\n===== Episode {e_idx}/{len(episodes)}: {len(epi_df)} samples =====")

    # Zero-shot evaluation on test set
    if len(test_df):
        history.append(evaluate_frame(student, test_df, tag=f'test_ep{e_idx}_zeroshot'))

    # Optional LwF snapshot (frozen teacher_prev)
    teacher_prev = None
    if USE_LWF:
        teacher_prev = tf.keras.models.clone_model(student)
        teacher_prev.build(student.inputs[0].shape)
        teacher_prev.set_weights(student.get_weights())
        teacher_prev.trainable = False

    # Build data loaders
    ds_l = build_labeled_ds(labeled_df, batch=BATCH_L, shuffle=True, repeat=True)
    ds_u = build_unlabeled_ds(epi_df, batch=BATCH_U)

    # Replay sampling (may be empty for ep1)
    replay_df = None if replay is None else replay.sample(int(REPLAY_FRACTION * BATCH_U * 4))
    ds_r = build_replay_ds(replay_df, batch=int(REPLAY_FRACTION * BATCH_U)) if (replay_df is not None) else None

    # Steps per epoch derived from unlabeled episode size
    steps_per_epoch = max(1, int(math.ceil(len(epi_df)/BATCH_U)))
    l_iter = iter(ds_l)

    # Local adaptation epochs
    for local_epoch in range(EPOCHS_ADAPT_PER_EP):
        tau = schedule_tau(local_epoch)
        lam_u = schedule_lambda_u(local_epoch)
        ema_a = schedule_ema(local_epoch)

        u_iter = iter(ds_u)
        for step in range(steps_per_epoch):
            # Labeled batch (repeat as needed)
            try: batch_l = next(l_iter)
            except StopIteration:
                l_iter = iter(ds_l); batch_l = next(l_iter)

            # Unlabeled batch
            try: batch_u = next(u_iter)
            except StopIteration:
                break

            # Optional replay batch
            batch_r = None
            if ds_r is not None:
                try:
                    r_iter = iter(ds_r)
                    batch_r = next(r_iter)
                except StopIteration:
                    batch_r = None

            # One training step and EMA update
            losses = train_step(student, teacher, batch_l, batch_u,
                                tau=tau, lambda_u=lam_u,
                                lambda_replay=LAMBDA_REPLAY,
                                use_lwf=USE_LWF, teacher_prev=teacher_prev,
                                replay_batch=batch_r)
            update_ema(teacher, student, alpha=ema_a)

        # Log per-epoch losses
        print(f"Episode {e_idx} epoch {local_epoch+1}:",
              {k: float(v.numpy() if hasattr(v, 'numpy') else v) for k,v in losses.items()})

    # Post-adaptation evaluation
    if len(test_df):
        history.append(evaluate_frame(student, test_df, tag=f'test_ep{e_idx}_adapted'))

    # Harvest confident pseudo-labels into replay buffer
    conf_df = collect_confident_pseudolabels(teacher, epi_df, tau=TAU_END)
    if len(conf_df):
        replay.add_many(conf_df)

    # Checkpoint both student and teacher (Keras format)
    os.makedirs('checkpoints_lifelong', exist_ok=True)
    student.save(f'checkpoints_lifelong/student_ep{e_idx}.keras')
    teacher.save(f'checkpoints_lifelong/teacher_ep{e_idx}.keras')

# Summarize and persist episode-wise metrics
hist_df = pd.DataFrame(history)
display(hist_df)
hist_path = 'lifelong_history.csv'
hist_df.to_csv(hist_path, index=False)
print('Saved:', hist_path)

{'tag': 'val_before_ep1', 'AUROC': 0.9999984845224275, 'PR-AUC': 0.9999992451738176, 'FPR@TPR=95%': 0.0, 'ECE': 0.3332943328182939}

===== Episode 1/4: 30171 samples =====


c:\Users\evri\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


{'tag': 'test_ep1_zeroshot', 'AUROC': 0.9999607467996805, 'PR-AUC': 0.9999846556212708, 'FPR@TPR=95%': 0.0, 'ECE': 0.3332900639119586}
Episode 1 epoch 1: {'loss_sup': 8.031516699702479e-06, 'loss_unsup': 0.014822359196841717, 'loss_replay': 0.01057648379355669, 'loss_lwf': -4.1383846109965816e-05}
Episode 1 epoch 2: {'loss_sup': 0.0002116000687237829, 'loss_unsup': 0.022062866017222404, 'loss_replay': 0.0019566104747354984, 'loss_lwf': -4.133726179134101e-05}
Episode 1 epoch 3: {'loss_sup': 0.009269711561501026, 'loss_unsup': 0.002470728475600481, 'loss_replay': 0.0028797246050089598, 'loss_lwf': -1.0396735888207331e-05}
Episode 1 epoch 4: {'loss_sup': 0.00011110883497167379, 'loss_unsup': 0.0169821809977293, 'loss_replay': 0.0017452797619625926, 'loss_lwf': 4.082500254298793e-06}
{'tag': 'test_ep1_adapted', 'AUROC': 0.9999749659574557, 'PR-AUC': 0.9999893407403228, 'FPR@TPR=95%': 0.0, 'ECE': 0.3332096630527069}

===== Episode 2/4: 30170 samples =====
{'tag': 'test_ep2_zeroshot', 'AURO

,tag,AUROC,PR-AUC,FPR@TPR=95%,ECE
0,val_before_ep1,0.999998,0.999999,0.0,0.333294
1,test_ep1_zeroshot,0.999961,0.999985,0.0,0.333290
2,test_ep1_adapted,0.999975,0.999989,0.0,0.333210
3,test_ep2_zeroshot,0.999975,0.999989,0.0,0.333210
4,test_ep2_adapted,0.999970,0.999988,0.0,0.333176
5,test_ep3_zeroshot,0.999970,0.999988,0.0,0.333176
6,test_ep3_adapted,0.999986,0.999994,0.0,0.333212
7,test_ep4_zeroshot,0.999986,0.999994,0.0,0.333212
8,test_ep4_adapted,0.999971,0.999989,0.0,0.333403


Saved: lifelong_history.csv
